In [1]:
import numpy as np
import pandas as pd
import os
import glob

# import dataset

In [2]:
dataset_psy = 'dataset/Youtube01-Psy.csv'
dataset_ktp = 'dataset/Youtube02-KatyPerry.csv'
dataset_LMFAO = 'dataset/Youtube03-LMFAO.csv'

In [3]:
df_psy = pd.read_csv(dataset_psy)
df_ktp = pd.read_csv(dataset_ktp)
df_lmfao = pd.read_csv(dataset_LMFAO)

# Preparing Dataset

In [4]:
df = pd.concat([df_psy,df_ktp,df_lmfao],axis=0)

In [5]:
df.head()

,COMMENT_ID,AUTHOR,DATE,CONTENT,CLASS
0,LZQPQhLyRh80UYxNuaDWhIGQYNQ96IuCg-AYWqNPjpU,Julius NM,2013-11-07T06:20:48,"Huh, anyway check out this you[tube] channel: ...",1
1,LZQPQhLyRh_C2cTtd9MvFRJedxydaVW-2sNg5Diuo4A,adam riyati,2013-11-07T12:37:15,Hey guys check out my new channel and our firs...,1
2,LZQPQhLyRh9MSZYnf8djyk0gEF9BHDPYrrK-qCczIY8,Evgeny Murashkin,2013-11-08T17:34:21,just for test I have to say murdev.com,1
3,z13jhp0bxqncu512g22wvzkasxmvvzjaz04,ElNino Melendez,2013-11-09T08:28:43,me shaking my sexy ass on my channel enjoy ^_^ ﻿,1
4,z13fwbwp1oujthgqj04chlngpvzmtt3r3dw,GsMega,2013-11-10T16:05:38,watch?v=vtaRGgvGtWQ Check this out .﻿,1


we dont need the comment_id ,author and date . so we will delete it.

In [6]:
df.drop(columns=['COMMENT_ID','AUTHOR','DATE'],inplace=True)

In [7]:
df.head()

,CONTENT,CLASS
0,"Huh, anyway check out this you[tube] channel: ...",1
1,Hey guys check out my new channel and our firs...,1
2,just for test I have to say murdev.com,1
3,me shaking my sexy ass on my channel enjoy ^_^ ﻿,1
4,watch?v=vtaRGgvGtWQ Check this out .﻿,1


In [8]:
df.loc[df['CLASS']==0]

,CONTENT,CLASS
7,i turned it on mute as soon is i came on i jus...,0
16,I'm only checking the views﻿,0
20,i think about 100 millions of the views come f...,0
23,just checking the views﻿,0
28,I dont even watch it anymore i just come here ...,0
...,...,...
427,The best Song i saw ❤️❤️❤️❤️❤️❤️❤️❤️😍😍😍😍😍😍😍😘😘😘...,0
428,watch this with sound off!﻿,0
429,Looooooooove this song!!!!!!!!!:))))))﻿,0
434,love this song﻿,0


In [9]:
df['CLASS'].unique()

array([1, 0], dtype=int64)

In [10]:
from sklearn.model_selection import train_test_split

In [11]:
 def spam_or_real_count(dataset):
    print('dataset ammount for REAL: {}'.format(dataset.loc[dataset['CLASS']==1].count()))
    print('dataset ammount for SPAM:{}'.format(dataset.loc[dataset['CLASS']==0].count()))


In [12]:
def split_data(df):
    spam_data = df.loc[df['CLASS']==1]
    real_data = df.loc[df['CLASS']==0]
    
    spam_trainX,spam_testX,spam_trainY,spam_testY = train_test_split(spam_data['CONTENT'],spam_data['CLASS'],test_size=0.1)
    real_trainX,real_testX,real_trainY,real_testY = train_test_split(real_data['CONTENT'],real_data['CLASS'],test_size=0.1)
    
    trainX = spam_trainX.append(real_trainX,ignore_index=True)
    testX = spam_testX.append(real_testX,ignore_index=True)
    trainY = spam_trainY.append(real_trainY,ignore_index=True)
    testY = spam_testY.append(real_testY,ignore_index=True)
    
    frames1 = [trainX,trainY]
    frames2 = [testX,testY]
    
    traindf = pd.concat(frames1,axis=1)
    testdf = pd.concat(frames2,axis=1)
    
    traindf = traindf.sample(frac=1).reset_index(drop=True)
    testdf = testdf.sample(frac=1).reset_index(drop=True)
    
    trainX = traindf['CONTENT']
    trainY = traindf['CLASS']
    testX = testdf['CONTENT']
    testY = testdf['CLASS']
    
    
    return trainX,testX,trainY,testY

In [13]:
trainX,testX,trainY,testY = split_data(df) 

In [14]:
trainX

0       Take a break from Katie and help me reach 100 ...
1       HAHAA THIS DANCE IS TIGHTTTT<br /><br />I know...
2       ''Little Psy, only 5 months left.. Tumor in th...
3       I can't comprehend why this video has nearly 7...
4       Like if you came here too see how many views t...
                              ...                        
1018                    Check out this video on YouTube:﻿
1019                    Check out this video on YouTube:﻿
1020    The TREU DETECTIVE alternate ending! __ http:/...
1021     HOW MANY THUMBS UP FOR LOUIS SAVING THE DAY!?!?﻿
1022    The girl in the train who was dancing, her out...
Name: CONTENT, Length: 1023, dtype: object

In [15]:
testY

0      1
1      1
2      0
3      1
4      0
      ..
110    0
111    0
112    0
113    0
114    1
Name: CLASS, Length: 115, dtype: int64

# Processing Dataset

In [16]:
import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords
from nltk.stem.porter import *

stemmer = PorterStemmer()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\62819\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [17]:
import re
from bs4 import BeautifulSoup

def review_words(word):
    text = BeautifulSoup(word, "html.parser").get_text()
    text = re.sub(r"[^a-zA-Z0-9]", " ", text.lower()) # Convert to lower case
    words = text.split()
    words = [w for w in words if w not in stopwords.words("english")]
    words = [PorterStemmer().stem(w) for w in words]
    
    return words 

In [18]:
import pickle

cache_dir = os.path.join('cache','preprocessed_word')
os.makedirs(cache_dir,exist_ok=True)

def preprocess_data(train_data,test_data,train_label,test_label,cache_dir=cache_dir,cache_file="preprocessed_data.pkl"):
    
    cache_data = None
    
    #we check if the cache file is exist or not.
    if cache_file is not None:
        try:
            with open(os.path.join(cache_dir,cache_file),"rb") as f:
                cache_data = pickle.load(f)
            print("Read preprocessed data from cache file:",cache_file)
        except:
            pass
        
    if cache_data is None:
        
        word_train = [review_words(word) for word in train_data]
        word_test = [review_words(word) for word in test_data]
        
        if cache_file is not None:
            cache_Data = dict(train_data=word_train,test_data=word_test,train_label=train_label,test_label=test_label)
            
            with open(os.path.join(cache_dir,cache_file),"wb") as f:
                pickle.dump(cache_Data,f)
            print("preprocessed word has been written to cache file:",cache_file)
        
    else:
        
        train_data,test_data,train_label,test_label = (cache_data['train_data'],cache_data['test_data'],cache_data['train_label'],cache_data['test_label'])
        
    return train_data,test_data,train_label,test_label

In [19]:
train_data,test_data,train_label,test_label = preprocess_data(trainX,testX,trainY,testY)

Read preprocessed data from cache file: preprocessed_data.pkl


In [20]:
train_data[0]

['awesom']

In [21]:
test_label

0      1
1      1
2      1
3      1
4      0
      ..
110    1
111    1
112    1
113    1
114    0
Name: CLASS, Length: 115, dtype: int64

# Feature extraction

# extract Bag of Word

In [22]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
import joblib

def extract_Bag_of_Word(train_data,test_data,vocab_size=4000,cache_dir=cache_dir,cache_file="bow_feature.pkl"):
    
    cache_data = None
    if cache_file is not None:
        try:
            with open(os.path.join(cache_dir,cache_file),"rb") as f:
                cache_data = pickle.load(f)
            print("Read feature from cache file:",cache_file)
        except:
            pass 
        
    if cache_data is None:
        
        def dummy_fun(doc):
            return doc
        vectorizer = CountVectorizer(max_features=vocab_size,preprocessor=dummy_fun, tokenizer=dummy_fun)
        features_train = vectorizer.fit_transform(train_data).toarray()
        features_test = vectorizer.transform(test_data).toarray()
        
        if cache_file is not None:
            vocabulary = vectorizer.vocabulary_
            cache_data = dict(features_train = features_train, features_test = features_test, vocabulary = vocabulary)
            with open(os.path.join(cache_dir,cache_file),"wb") as f:
                joblib.dump(cache_data,f)
            print("wrote features to cache file:",cache_file)
            
    else:
        features_train ,features_test ,vocabulary = (cache_data['features_train'],cache_data['features_test'],
                                                    cache_data['vocabulary'])
        
    return features_train,features_test ,vocabulary
        
            

In [23]:
train_data ,test_data ,vocabulary = extract_Bag_of_Word(train_data,test_data)

wrote features to cache file: bow_feature.pkl


In [24]:
len(train_data)


1023

In [25]:
train_data[0]

array([0, 0, 0, ..., 0, 0, 0], dtype=int64)

In [26]:
train_data = pd.DataFrame(train_data[:200])
valid_data = pd.DataFrame(train_data[200:])
train_label = pd.DataFrame(train_label[:200])
valid_label = pd.DataFrame(train_label[200:])
test_data = pd.DataFrame(test_data)
test_label = pd.DataFrame(test_label)

In [27]:
folder_name = 'train_test_dataset'
os.makedirs(folder_name,exist_ok=True)

var_str = ['train_data','valid_data','train_label','valid_label','test_data','test_label']
var_name = [train_data,valid_data,train_label,valid_label,test_data,test_label]
for df,file_name in zip(var_name,var_str):
    df.to_csv(os.path.join(folder_name,'{}.csv'.format(file_name)), header = False, index=False)